# MapReduce Mini‑Projects — Solutions
*Generated: 2025-08-25 04:50*

In [1]:

# ---- Sample Datasets (reuse across tasks) ----

# Tiny web pages corpus
web_pages = [
    "MapReduce makes big data processing scalable",
    "Map reduce reduces the complexity of processing",
    "Big data needs big ideas",
]

# LinkedIn-like view logs: (user_id, content_type, content_id, timestamp)
logs = [
    ("u1", "profile", "p10", "2025-08-20 09:00:01"),
    ("u2", "post",    "c42", "2025-08-20 09:00:05"),
    ("u1", "post",    "c42", "2025-08-20 09:01:10"),
    ("u3", "article", "a7",  "2025-08-20 09:02:00"),
    ("u2", "profile", "p10", "2025-08-20 09:03:10"),
    ("u4", "video",   "v3",  "2025-08-20 09:03:21"),
    ("u3", "post",    "c42", "2025-08-20 09:04:00"),
    ("u1", "video",   "v3",  "2025-08-20 09:05:10"),
    ("u5", "post",    "c9",  "2025-08-20 09:06:30"),
    ("u2", "video",   "v3",  "2025-08-20 09:07:11"),
]

# Profiles: user -> headline/skills text
profiles = {
    "u1": "Data Scientist | Python | Spark | ML",
    "u2": "Backend Engineer | Go | Kubernetes | Cloud",
    "u3": "ML Engineer | Python | MLOps | Cloud",
    "u4": "Frontend Developer | React | TypeScript",
    "u5": "Data Analyst | SQL | Python | BI",
}

# Social graph edges (connections are undirected)
edges = [
    ("u1","u2"), ("u2","u3"), ("u3","u4"), ("u1","u5"),
    # note: u2-u5 not connected, u4-u5 not connected
]


In [2]:

from collections import defaultdict
def shuffle_group(pairs):
    grouped = defaultdict(list)
    for k, v in pairs:
        grouped[k].append(v)
    return grouped


## Solution 1 — Word Count with Stop‑Words

In [3]:

stop = {'the','and','of','a','an','to','is','big'}

def map_words(pages):
    pairs = []
    for page in pages:
        for w in page.lower().split():
            w = ''.join(ch for ch in w if ch.isalpha())  # keep letters
            if not w or w in stop: 
                continue
            pairs.append((w, 1))
    return pairs

def reduce_counts(grouped):
    return {w: sum(vs) for w, vs in grouped.items()}

pairs = map_words(web_pages)
wc = reduce_counts(shuffle_group(pairs))
sorted(wc.items())


[('complexity', 1),
 ('data', 2),
 ('ideas', 1),
 ('makes', 1),
 ('map', 1),
 ('mapreduce', 1),
 ('needs', 1),
 ('processing', 2),
 ('reduce', 1),
 ('reduces', 1),
 ('scalable', 1)]

## Solution 2 — Views per Content Type

In [4]:

def map_types(rows):
    return [(ctype, 1) for (_, ctype, _, _) in rows]

def reduce_sum(grouped):
    return {k: sum(vs) for k,vs in grouped.items()}

views_by_type = reduce_sum(shuffle_group(map_types(logs)))
views_by_type


{'profile': 2, 'post': 4, 'article': 1, 'video': 3}

## Solution 3 — Top‑K Content by Views (K=2)

In [5]:

def map_content(rows):
    return [(cid, 1) for (_,_,cid,_) in rows]

def top_k(counts, k=2):
    return sorted(counts.items(), key=lambda x: (-x[1], x[0]))[:k]

counts = {k: sum(vs) for k,vs in shuffle_group(map_content(logs)).items()}
top_k(counts, 2)


[('c42', 3), ('v3', 3)]

## Solution 4 — Inverted Index of Skills → Users

In [6]:

import re
def tokenize(text):
    return [t.lower() for t in re.findall(r"[A-Za-z]+", text)]

def map_profile_tokens(profiles):
    pairs = []
    for uid, text in profiles.items():
        for tok in tokenize(text):
            pairs.append((tok, uid))
    return pairs

def reduce_unique_users(grouped):
    out = {}
    for tok, users in grouped.items():
        out[tok] = sorted(set(users))
    return out

index = reduce_unique_users(shuffle_group(map_profile_tokens(profiles)))
{k: index[k] for k in list(index)[:8]}


{'data': ['u1', 'u5'],
 'scientist': ['u1'],
 'python': ['u1', 'u3', 'u5'],
 'spark': ['u1'],
 'ml': ['u1', 'u3'],
 'backend': ['u2'],
 'engineer': ['u2', 'u3'],
 'go': ['u2']}

## Solution 5 — Friend‑of‑Friend Recommendations (Mutuals)

In [7]:

from collections import defaultdict
from itertools import combinations

def neighbors(edges):
    nb = defaultdict(set)
    for a,b in edges:
        nb[a].add(b); nb[b].add(a)
    return nb

def map_mutuals(edges):
    nb = neighbors(edges)
    pairs = []
    for x, nbrs in nb.items():
        for a, b in combinations(sorted(nbrs), 2):
            pairs.append(((a,b), 1))
    return pairs

def reduce_counts(grouped):
    return {k: sum(vs) for k,vs in grouped.items()}

def suggestions_for(edges, topn=2):
    nb = neighbors(edges)
    counts = reduce_counts(shuffle_group(map_mutuals(edges)))
    sugg = defaultdict(list)
    for (a,b), c in counts.items():
        if b not in nb[a]: 
            sugg[a].append((b,c))
        if a not in nb[b]:
            sugg[b].append((a,c))
    for u in list(sugg):
        sugg[u] = sorted(sugg[u], key=lambda x: (-x[1], x[0]))[:topn]
    return dict(sugg)

suggestions_for(edges, topn=2)


{'u2': [('u4', 1), ('u5', 1)],
 'u5': [('u2', 1)],
 'u1': [('u3', 1)],
 'u3': [('u1', 1)],
 'u4': [('u2', 1)]}

### End of solutions